# Free image → video with Wan 2.2 (ComfyUI) on Kaggle

Turns one picture into a short animated clip using **Wan 2.2 TI2V 5B**, free on Kaggle's GPU.

## One-time setup (Kaggle settings, right-hand panel)
1. **Session options → Accelerator → GPU T4 x2** (or P100).
2. **Session options → Internet → On.** (Kaggle asks you to verify your phone number once. No credit card.)
3. **Your image:** click **+ Add Input → Upload → New Dataset**, upload your PNG/JPG. It will appear under `/kaggle/input/...`.
   Or paste a direct image link into `IMAGE_URL` in step 3.

## How to use
- **First time:** Run All. Steps 1–2 install everything (~10 min, downloads ~17 GB of models).
- **Each new clip:** edit step 3 (image + prompt), then run steps 3 and 4 only.
- Your video appears at the bottom and is saved to `/kaggle/working/` (Output tab → download).

Expect roughly **5–15 minutes per clip** on a T4 at the default size.

## Step 1 — Install ComfyUI

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/comfyanonymous/ComfyUI.git
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 2 — Download Wan 2.2 models (~17 GB) and start ComfyUI

In [ ]:
import os, subprocess, time, urllib.request

C = "/kaggle/working/ComfyUI"
MODELS = {
    f"{C}/models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors":
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
    f"{C}/models/vae/wan2.2_vae.safetensors":
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors",
    f"{C}/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
}
for path, url in MODELS.items():
    if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
        print("already have", os.path.basename(path)); continue
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print("downloading", os.path.basename(path), "...")
    subprocess.run(["wget", "-q", "--show-progress", "--progress=bar:force:noscroll", "-O", path, url], check=True)

# Start ComfyUI in the background on GPU 0
log = open("/kaggle/working/comfyui.log", "w")
server = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd=C, stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI did not start — see /kaggle/working/comfyui.log")
print("ComfyUI is running ✅")

## Step 3 — Your image and prompt ✏️
Describe the **motion and camera**, not just the picture. Example for the 2.5D cartoon style:
*"The cartoon character waves and smiles, camera slowly pushes in, background town gently moves with parallax, smooth 3D animation."*

In [ ]:
IMAGE_PATH = ""   # e.g. "/kaggle/input/my-images/scene1.png"  (leave "" to use IMAGE_URL)
IMAGE_URL  = ""   # or a direct link to a .png/.jpg

PROMPT = ("The cartoon character waves happily and smiles, the camera slowly pushes in, "
          "the colorful background moves gently with depth, smooth stylized 3D animation, high quality")
NEGATIVE = ("blurry, distorted face, extra limbs, deformed hands, flicker, jitter, text, watermark, "
            "low quality, static, frozen frame")

# Vertical 9:16 for Shorts/TikTok. Must be multiples of 32. For landscape swap them.
WIDTH, HEIGHT = 544, 960
SECONDS = 3            # clip length (Wan 2.2 5B runs at 24 fps). 3–5 s recommended on a T4.
STEPS   = 20           # 20 = faster, 30 = a bit better quality
CFG     = 5.0
SEED    = 42           # change for a different take
NAME    = "clip"       # output file name

## Step 4 — Generate the video 🎬

In [ ]:
import json, os, random, shutil, glob, time, urllib.request, subprocess
from IPython.display import Video, display

C = "/kaggle/working/ComfyUI"
FPS = 24
length = int(SECONDS * FPS) // 4 * 4 + 1   # Wan needs 4n+1 frames

# Put the input image into ComfyUI/input
os.makedirs(f"{C}/input", exist_ok=True)
in_name = f"{NAME}_input.png"
if IMAGE_PATH:
    shutil.copy(IMAGE_PATH, f"{C}/input/{in_name}")
elif IMAGE_URL:
    urllib.request.urlretrieve(IMAGE_URL, f"{C}/input/{in_name}")
else:
    raise ValueError("Set IMAGE_PATH or IMAGE_URL in step 3")

seed = SEED if SEED is not None else random.randint(0, 2**31)
prefix = f"{NAME}_{int(time.time())}"
workflow = {
  "1": {"class_type": "UNETLoader", "inputs": {"unet_name": "wan2.2_ti2v_5B_fp16.safetensors", "weight_dtype": "default"}},
  "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}},
  "3": {"class_type": "VAELoader",  "inputs": {"vae_name": "wan2.2_vae.safetensors"}},
  "4": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["1", 0], "shift": 8.0}},
  "5": {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT,   "clip": ["2", 0]}},
  "6": {"class_type": "CLIPTextEncode", "inputs": {"text": NEGATIVE, "clip": ["2", 0]}},
  "7": {"class_type": "LoadImage", "inputs": {"image": in_name}},
  "8": {"class_type": "Wan22ImageToVideoLatent", "inputs": {"vae": ["3", 0], "width": WIDTH, "height": HEIGHT,
        "length": length, "batch_size": 1, "start_image": ["7", 0]}},
  "9": {"class_type": "KSampler", "inputs": {"model": ["4", 0], "seed": seed, "steps": STEPS, "cfg": CFG,
        "sampler_name": "uni_pc", "scheduler": "simple", "positive": ["5", 0], "negative": ["6", 0],
        "latent_image": ["8", 0], "denoise": 1.0}},
  "10": {"class_type": "VAEDecode", "inputs": {"samples": ["9", 0], "vae": ["3", 0]}},
  "11": {"class_type": "SaveImage", "inputs": {"images": ["10", 0], "filename_prefix": prefix}},
}

req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=json.dumps({"prompt": workflow}).encode(),
                             headers={"Content-Type": "application/json"})
try:
    pid = json.load(urllib.request.urlopen(req))["prompt_id"]
except urllib.error.HTTPError as e:
    raise RuntimeError(e.read().decode())
print(f"Generating {length} frames ({length/FPS:.1f}s) at {WIDTH}x{HEIGHT}, seed {seed} … this takes several minutes")

t0 = time.time()
while True:
    hist = json.load(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}"))
    if pid in hist:
        status = hist[pid].get("status", {})
        if status.get("status_str") == "error":
            raise RuntimeError(json.dumps(status.get("messages", [])[-1:], indent=1))
        break
    time.sleep(10)
    print(f"  … {int(time.time()-t0)}s", end="\r")

# Join frames into an MP4
frames = sorted(glob.glob(f"{C}/output/{prefix}_*.png"))
out = f"/kaggle/working/{prefix}.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-framerate", str(FPS),
                "-i", f"{C}/output/{prefix}_%05d_.png",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", out], check=True)
for f in frames: os.remove(f)
print(f"\nDone in {int(time.time()-t0)}s → {out}  (download from the Output tab)")
display(Video(out, embed=True, width=360))

## Tips
- **Out of memory?** Lower `WIDTH, HEIGHT` to `480, 832` or `SECONDS` to 2.
- **Bad take?** Change `SEED` and run step 4 again.
- **Longer scenes:** make several 3–5 s clips and join them in CapCut, as in the reference shorts.
- **Session ended?** Kaggle wipes the machine; just Run All again (models re-download).
- Free GPU quota is about **30 hours/week** — check it under your Kaggle profile.